<a href="https://colab.research.google.com/github/trankhactue/fall/blob/main/trankhactue-202625099Untitled5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 데이터 탐색 및 전처리
### 1-1. Titanic 데이터 불러오기 및 결측치 확인
실습 자료에 포함된 데이터 주소를 사용하여 데이터를 불러오고, 데이터 크기, 결측치 비율 및 타겟 변수(`Survived`) 분포를 확인한다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/wesm/pydata-book/3rd-edition/datasets/titanic/"
df = pd.read_csv(DATA_URL + "train.csv")

print("데이터 크기:", df.shape)
display(df.head())
df.info()

# 결측치 현황 확인
missing_df = pd.DataFrame({
    "결측치_수": df.isna().sum(),
    "결측치_비율(%)": (df.isna().mean() * 100).round(2)
})
print("\n[결측치 현황]")
display(missing_df[missing_df["결측치_수"] > 0])

# 타겟 변수 분포 확인
print("\n[Survived 분포]")
print(df["Survived"].value_counts())
print((df["Survived"].value_counts(normalize=True) * 100).round(2))

#2. Feature Engineering 및 인코딩 결과

In [ ]:
def extract_title(name: str) -> str:
    title = name.split(",")[1].split(".")[0].strip()
    female_titles = {"Miss", "Mrs", "Ms", "Mlle", "Lady", "Mme", "the Countess", "Dona"}
    special_male = {"Dr", "Col", "Major", "Jonkheer", "Capt", "Sir", "Don", "Rev"}
    if title in female_titles:
        return "Female_Title"
    elif title == "Master":
        return "Master"
    elif title in special_male:
        return "Special_Male"
    else:
        return "Mr"

def categorize_family(size: int) -> str:
    if size == 1:
        return "Alone"
    elif 2 <= size <= 4:
        return "Small"
    elif 5 <= size <= 6:
        return "Medium"
    else:
        return "Large"

def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    df_new = data.copy()
    df_new["Cabin_Has"] = df_new["Cabin"].notna().astype(int)
    df_new = df_new.drop(columns=["Cabin"])
    df_new["Family_Size"] = df_new["SibSp"] + df_new["Parch"] + 1
    df_new["Family_Group"] = df_new["Family_Size"].apply(categorize_family)
    df_new["Title"] = df_new["Name"].apply(extract_title)
    drop_cols = [c for c in ["PassengerId", "Name", "Ticket"] if c in df_new.columns]
    return df_new.drop(columns=drop_cols)

df_fe = engineer_features(df)
X = df_fe.drop(columns=["Survived"])
y = df_fe["Survived"].astype(int)

print("Feature Engineering 후 데이터 크기:", X.shape)
display(X[["Family_Size", "Family_Group", "Title", "Cabin_Has"]].head())

In [ ]:
preview_cat_cols = ["Sex", "Embarked", "Pclass", "Family_Group", "Title"]
ohe_preview = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
cat_imputed = SimpleImputer(strategy="most_frequent").fit_transform(X[preview_cat_cols])
encoded_arr = ohe_preview.fit_transform(cat_imputed)
encoded_df = pd.DataFrame(encoded_arr, columns=ohe_preview.get_feature_names_out(preview_cat_cols))

print("원핫 인코딩 후 shape:", encoded_df.shape)
display(encoded_df.head())

#3 .KNN 모델 학습 및 5-Fold 교차검증 결과

In [ ]:
class GroupAgeImputer(BaseEstimator, TransformerMixin):
    def fit(self, X_df, y=None):
        temp = pd.DataFrame(X_df).copy()
        self.global_median_ = temp["Age"].median()
        self.group_medians_ = temp.groupby(["Pclass", "Sex"])["Age"].median()
        return self

    def transform(self, X_df):
        temp = pd.DataFrame(X_df).copy()
        for (pclass, sex), med in self.group_medians_.items():
            mask = temp["Age"].isna() & (temp["Pclass"] == pclass) & (temp["Sex"] == sex)
            temp.loc[mask, "Age"] = med
        temp["Age"] = temp["Age"].fillna(self.global_median_)
        return temp[["Age"]].to_numpy()

age_cols = ["Age", "Pclass", "Sex"]
num_cols = ["Fare", "SibSp", "Parch", "Family_Size", "Cabin_Has"]
cat_cols = ["Sex", "Embarked", "Pclass", "Family_Group", "Title"]

preprocessor = ColumnTransformer([
    ("age", Pipeline([("group_age", GroupAgeImputer()), ("scaler", StandardScaler())]), age_cols),
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols)
], remainder="drop")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

base_knn_pipe = Pipeline([
    ("prep", preprocessor),
    ("knn", KNeighborsClassifier())
])

base_scores = cross_val_score(base_knn_pipe, X, y, cv=skf, scoring="accuracy", n_jobs=-1)
print("Fold별 Accuracy:", np.round(base_scores, 4))
print(f"기본 모델 평균 Accuracy: {base_scores.mean():.4f}")
print(f"표준편차: {base_scores.std():.4f}")

#4. 하이퍼파라미터 튜닝 및 성능 평가 (GridSearchCV)

In [ ]:
param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]
}

grid_search = GridSearchCV(
    estimator=base_knn_pipe,
    param_grid=param_grid,
    cv=skf,
    scoring="accuracy",
    n_jobs=-1,
    refit=True
)

grid_search.fit(X, y)

best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("최적 하이퍼파라미터:", best_params)
print(f"최적 모델 평균 Accuracy: {best_score:.4f}")

#5. 튜닝 전후 Accuracy 비교

In [ ]:
base_mean = base_scores.mean()
diff = best_score - base_mean

comparison_df = pd.DataFrame({
    "모델": ["튜닝 전 (기본 KNN)", "튜닝 후 (최적 KNN)"],
    "5-Fold 평균 Accuracy": [round(base_mean, 4), round(best_score, 4)],
    "하이퍼파라미터": ["n_neighbors=5, weights='uniform', p=2", str(best_params)]
})

display(comparison_df)
print(f"Accuracy 변화량 (튜닝 후 - 튜닝 전): {diff:+.4f}")

#6. 최종 결과에 대한 간단한 해석 (3~5줄)

In [ ]:
print("1. Age 결측치를 Pclass 및 Sex 그룹별 중앙값으로 대체하고 Family_Size, Family_Group, Title, Cabin_Has 파생변수 생성 후 원핫 인코딩과 StandardScaler를 적용해 거리 기반 KNN의 스케일 차이를 줄였습니다.")
print("2. 전처리와 스케일링을 Pipeline 내부에 포함하여 5-Fold Stratified 교차검증 및 GridSearchCV 과정의 데이터 누수(Data Leakage)를 방지했습니다.")
print(f"3. GridSearchCV 탐색 결과 기본 KNN 평균 Accuracy({base_scores.mean():.4f})에서 최적 모델({grid_search.best_params_}) 기준 {grid_search.best_score_:.4f}로 성능이 변화했으며, 이웃 수와 거리 가중치 설정이 분류 성능에 영향을 줍니다.")